# 4-1 RAG 기반 
LLM의 한계를 보완. 키워드 검색 -> 의미 기반 검색 -> 청킹 전략 -> LangGraph StateGraph로 RAG 파이프라인 구현 

  - Step 1: 환경 설정 — 라이브러리 설치 및 API Key 설정
  - Step 2: LLM의 한계 체감 - 환각 현상
  - Step 3: 자료와 함께하면 해결 - 자료를 직접 프롬프트에 넣어 질문. 컨텍스트 기반 답변.
  - Step 4: 키워드로 문서 검색 - 정확히 일치하는 단어만 찾는 한계점 
  - Step 5: 의미 기반 검색 + DB화 기초 - 문서를 vector DB에 저장(임베딩), 의미 기반 검색 수행
  - Step 6: 청킹 전략 - chunking, chunnk_size/overlap 트레이드 오프
  - Step 7: 전체 RAG 파이프라인 (LangGraph) 구현 

## Step 2: LLM의 한계 체감 - 환각 현상


In [ ]:
# TODO: LLM에게 직접 질문하기
from langchain_core.messages import HumanMessage

# Yes24 서비스에 대한 질문
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

# LLM에게 직접 질문
response = llm.invoke([HumanMessage(content=question)]) #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

## Step 3: 자료와 함께하면 해결

In [ ]:
# TODO 2: 자료를 직접 프롬프트에 넣어 질문하기
from langchain_community.document_loaders import PyMuPDFLoader

# 총알배송 문서 로드
pdf_path = DATA_DIR + "총알배송_서비스 혜택 - 예스24.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

# 문서 내용 확인
doc_content = "\n".join([doc.page_content for doc in documents])
print(f"문서 내용 (일부):\n{doc_content[:500]}...")


# 컨텍스트를 포함한 프롬프트 구성
from langchain_core.prompts import ChatPromptTemplate

#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 온라인 서점의 고객 서비스 상담원입니다.
다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다"라고 답변하세요.

[참고 자료]
{context}"""),
    ("human", "{question}")
])#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 질문
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

# 프롬프트 생성 후 LLM 직접 호출
messages = prompt_template.format_messages(context=doc_content, question=question)
response = llm.invoke(messages)
print(response.content)


## Step 4: 키워드로 문서 검색

In [ ]:
# TODO: 키워드로 관련 문서 검색하기

# 모든 PDF 문서 로드
all_documents = []
for pdf_path in pdf_files:
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    # 파일명을 메타데이터에 추가
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

print(f"📚 총 {len(all_documents)}개 페이지 로드 완료")

# 키워드 검색 함수 #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
def keyword_search(documents, keyword):
    """문서 리스트에서 키워드를 포함한 문서 검색"""
    results = []
    for doc in documents:
        if keyword in doc.page_content:
            results.append(doc)
    return results
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 키워드 검색 테스트
keyword = "총알배송"
results = keyword_search(all_documents, keyword)
print(f"🔍 키워드 '{keyword}' 검색 결과: {len(results)}개 문서")
for doc in results[:2]:
    print(f"  - {doc.metadata.get('source_file', 'Unknown')}")


## Step 5: 의미 기반 검색 + DB화 기초
임베딩과 vector store
코사인 유사도 사용.

In [ ]:
# TODO: 문서를 Vector DB에 저장하기
from langchain_community.vectorstores import Chroma
import tiktoken

# 임베딩 모델의 토큰 제한 처리 (text-embedding-3-small: 최대 8,191 토큰)
# PDF 전체 페이지는 토큰 제한을 초과할 수 있으므로 사전에 잘라줌
MAX_TOKENS = 8000  # 안전 여유분 확보
enc = tiktoken.encoding_for_model("text-embedding-3-small")

truncated_count = 0
for doc in all_documents:
    tokens = enc.encode(doc.page_content)
    if len(tokens) > MAX_TOKENS:
        doc.page_content = enc.decode(tokens[:MAX_TOKENS])
        truncated_count += 1

if truncated_count > 0:
    print(f"⚠️ {truncated_count}개 문서가 토큰 제한 초과로 잘림 (최대 {MAX_TOKENS} 토큰)")

# Vector Store 생성 (문서 단위 - 페이지별)#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name="yes24_docs_page"
)#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

print(f"✅ Vector Store 생성 완료")
print(f"📊 저장된 문서 수: {vectorstore._collection.count()}개")

In [ ]:
# TODO: 의미 기반 검색 수행하기

# Retriever 생성#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 의미 기반 검색 테스트
test_queries = [
    "빠른 배송",      # 총알배송과 유사한 의미
    "배송 빨리",      # 총알배송과 유사한 의미  
    "신속 배송",      # 총알배송과 유사한 의미
    "총알배송",       # 정확한 키워드
]

print("🔍 의미 기반 검색 테스트:")
for query in test_queries:
    results = retriever.invoke(query)
    source_files = set([doc.metadata.get('source_file', 'Unknown') for doc in results])
    print(f"  '{query}': {source_files}")


## Step 6: 청킹 전략

In [ ]:
# TODO: 청킹 전략 비교하기
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 다양한 청킹 설정 비교
chunking_configs = [
    {"chunk_size": 100, "chunk_overlap": 0},
    {"chunk_size": 200, "chunk_overlap": 20},
    {"chunk_size": 500, "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 100},
]

# 하나의 문서로 테스트
test_doc = all_documents[0]
print(f"📄 원본 문서 길이: {len(test_doc.page_content)} 글자\n")

for config in chunking_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        length_function=len,
    )
    chunks = splitter.split_documents([test_doc])
    print(f"chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}: {len(chunks)}개 청크")

# 최적 청킹 설정 선택 및 전체 문서 청킹#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 전체 문서 청킹
chunked_documents = text_splitter.split_documents(all_documents)

print(f"📊 청킹 결과:")
print(f"  - 원본 문서 수: {len(all_documents)}개")
print(f"  - 청킹 후 조각 수: {len(chunked_documents)}개")
print(f"  - 평균 청크 크기: {sum(len(doc.page_content) for doc in chunked_documents) / len(chunked_documents):.0f}자")

# 청킹된 문서로 새 Vector Store 생성
vectorstore_chunked = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name="yes24_docs_chunked"
)

retriever_chunked = vectorstore_chunked.as_retriever(search_kwargs={"k": 3})

print(f"✅ 청킹된 문서로 Vector Store 재생성 완료")
print(f"📊 저장된 청크 수: {vectorstore_chunked._collection.count()}개")


## Step 7: 전체 RAG 파이프라인 (LangGraph) 구현
LangChain 팀에서 개발한 **상태 기반 워크플로우 프레임워크**입니다.
명시적 상태 관리: 각 노드가 상태를 읽고 쓰며, 데이터 흐름이 명확함
**LangGraph의 핵심 구성요소:**
- **State**: 그래프 전체에서 공유되는 데이터 구조
- **Node**: 상태를 변환하는 함수 (검색, 생성 등)
- **Edge**: 노드 간 연결 (순차 실행, 조건부 분기)
[LangGraph RAG 파이프라인 구조]


        START
          │
          ▼
    ┌─────────────┐
    │   retrieve  │  ← 질문으로 관련 문서 검색
    │   (Node 1)  │
    └─────────────┘
          │
          ▼
    ┌─────────────┐
    │   generate  │  ← 검색된 문서로 답변 생성
    │   (Node 2)  │
    └─────────────┘
          │
          ▼
         END

In [ ]:
# TODO: LangGraph StateGraph로 RAG 파이프라인 구현
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# 1. State 타입 정의
# - question: 사용자 질문
# - context: 검색된 문서 내용
# - answer: 생성된 답변
class RAGState(TypedDict):
    question: str
    context: str
    answer: str

# 2. retrieve 노드: 질문으로 관련 문서 검색#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
def retrieve(state: RAGState) -> RAGState:
    """Vector Store에서 관련 문서를 검색하는 노드"""
    question = state["question"]
    
    # 청킹된 Vector Store에서 검색
    docs = retriever_chunked.invoke(question)
    
    # 검색 결과를 문자열로 포맷팅
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return {"context": context}#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 3. generate 노드: 검색된 문서로 답변 생성#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드"""
    question = state["question"]
    context = state["context"]
    
    # RAG 프롬프트 정의
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 AI 온라인 서점 'Yes24'의 고객 서비스 상담원입니다.
다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.

[검색된 자료]
{context}

답변 규칙:
1. 검색된 자료를 기반으로 답변하세요
2. 자료에 없는 내용은 추측하지 마세요
3. 존댓말을 사용하세요
4. 간결하고 명확하게 답변하세요"""),
        ("human", "{question}")
    ])
    
    # 프롬프트 생성 후 LLM 직접 호출
    messages = rag_prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    answer = response.content
    
    return {"answer": answer}#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 4. StateGraph 구성
# - 노드 추가: retrieve, generate
# - 엣지 연결: START → retrieve → generate → END
workflow = StateGraph(RAGState)#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 노드 추가#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

# 엣지 연결 (순차 실행)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 그래프 컴파일#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
rag_graph = workflow.compile()

print("✅ LangGraph RAG 파이프라인 구성 완료!")

# LangGraph RAG 파이프라인 테스트
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

print("📝 질문:", question)
print("\n🤖 LangGraph RAG 응답:")

# 그래프 실행
result = rag_graph.invoke({"question": question})
print(result["answer"])
